# MODIFIED FOR RENDER DEPLOYMENT



In [1]:
# Cell 1: Mount Google Drive (for persistent storage)
from google.colab import drive
drive.mount('/content/drive')

# Create working directories
!mkdir -p /content/model-service/models
!mkdir -p /content/model-service/logs
!mkdir -p /content/model-service/cache

# Cell 2: Install Dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate bitsandbytes peft
!pip install -q fastapi uvicorn nest-asyncio
!pip install -q python-multipart
!pip install -q aiofiles python-dotenv
!pip install -q sentencepiece protobuf
!pip install -q gunicorn

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 7.5 MB/s eta 0:00:00


In [2]:
# Cell 3: Import Libraries
import torch
import torch.nn as nn
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    GenerationConfig
)
import logging
import json
import os
import time
import uuid
import asyncio
import threading
from typing import Optional, Dict, Any, List
from dataclasses import dataclass
import nest_asyncio
from pathlib import Path
from fastapi import FastAPI, HTTPException, BackgroundTasks, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse, JSONResponse
from pydantic import BaseModel, Field
import uvicorn

# Apply nest_asyncio
nest_asyncio.apply()

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [3]:
# Cell 4: Login to Hugging Face
from huggingface_hub import login
from google.colab import userdata

# Retrieve your token from Colab secrets
try:
    hf_token = userdata.get('HF_Token')
    if hf_token:
        login(token=hf_token)
        print("✅ Successfully logged in to Hugging Face!")
    else:
        print("❌ HF_TOKEN not found in Colab secrets.")
except Exception as e:
    print(f"❌ Could not retrieve HF_TOKEN: {e}")

✅ Successfully logged in to Hugging Face!


In [4]:
# Cell 5: Download Model
from huggingface_hub import snapshot_download

# Your model ID
model_name = "ugonna/llama3.18B-Fine-tunedByUgo3"

print(f"Downloading model: {model_name}")
print("This may take several minutes...")

try:
    snapshot_download(
        repo_id=model_name,
        local_dir="/content/model-service/models",
        local_dir_use_symlinks=False,
        ignore_patterns=["*.h5", "*.ot", "*.msgpack"]
    )
    print("✅ Model downloaded successfully!")
except Exception as e:
    print(f"❌ Download failed: {e}")
    raise

This may take several minutes...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

✅ Model downloaded successfully!


**create the Model Manager Class**

In [5]:
# Cell 6: Model Manager Class
class ModelManager:
    def __init__(self):
        self.model = None
        self.tokenizer = None
        self.device = None
        self.model_path = "/content/model-service/models"
        self.loading_status = "not_started"
        self.loading_progress = 0
        self.loading_error = None

    def setup_device(self):
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
            gpu_name = torch.cuda.get_device_name(0)
            gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
            logger.info(f"Using GPU: {gpu_name} ({gpu_memory:.1f} GB)")
            torch.cuda.empty_cache()
        else:
            self.device = torch.device("cpu")
            logger.info("Using CPU")

    def load_model(self, use_4bit: bool = True):
        """Load model with progress tracking"""
        try:
            self.loading_status = "loading"
            self.setup_device()

            logger.info("Starting model loading...")

            # Configure quantization
            if use_4bit and self.device.type == "cuda":
                quantization_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_quant_type="nf4"
                )
                logger.info("Using 4-bit quantization")
            else:
                quantization_config = None

            # Load tokenizer
            logger.info("Loading tokenizer...")
            self.tokenizer = AutoTokenizer.from_pretrained(
                self.model_path,
                trust_remote_code=True
            )

            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token

            self.loading_progress = 30
            logger.info("✓ Tokenizer loaded")

            # Load model
            logger.info("Loading model (this will take several minutes)...")
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_path,
                quantization_config=quantization_config,
                torch_dtype=torch.float16 if self.device.type == "cuda" else torch.float32,
                device_map="auto" if self.device.type == "cuda" else None,
                trust_remote_code=True,
                low_cpu_mem_usage=True
            )

            self.loading_progress = 90

            if self.device.type != "cuda":
                self.model = self.model.to(self.device)

            self.loading_progress = 100
            self.loading_status = "loaded"
            logger.info("✅ Model loaded successfully!")

            # Quick warmup
            self._warmup()

            return True

        except Exception as e:
            self.loading_status = "failed"
            self.loading_error = str(e)
            logger.error(f"Failed to load model: {e}")
            import traceback
            traceback.print_exc()
            return False

    def _warmup(self):
        """Quick warmup to ensure model is ready"""
        try:
            logger.info("Warming up model...")
            test_input = "Hello"
            inputs = self.tokenizer(test_input, return_tensors="pt")
            if self.device.type == "cuda":
                inputs = {k: v.to(self.device) for k, v in inputs.items()}
            with torch.no_grad():
                _ = self.model.generate(**inputs, max_new_tokens=5)
            logger.info("✓ Warmup complete")
        except Exception as e:
            logger.warning(f"Warmup failed: {e}")

    def generate(self, prompt: str, **kwargs) -> Dict[str, Any]:
        if self.model is None:
            raise Exception("Model not loaded")

        start_time = time.time()

        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)

        if self.device.type == "cuda":
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=kwargs.get('max_tokens', 512),
                temperature=kwargs.get('temperature', 0.7),
                top_p=kwargs.get('top_p', 0.9),
                top_k=kwargs.get('top_k', 50),
                repetition_penalty=kwargs.get('repetition_penalty', 1.1),
                do_sample=True,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )

        generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
        generated_text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)

        inference_time = time.time() - start_time
        tokens_generated = len(generated_ids)

        return {
            "text": generated_text,
            "usage": {
                "prompt_tokens": inputs['input_ids'].shape[1],
                "completion_tokens": tokens_generated,
                "total_tokens": inputs['input_ids'].shape[1] + tokens_generated,
                "inference_time": inference_time,
                "tokens_per_second": tokens_generated / inference_time if inference_time > 0 else 0
            }
        }

In [6]:
# Cell 7: Pydantic Models
class GenerationRequest(BaseModel):
    prompt: str = Field(..., description="Input prompt")
    max_tokens: int = Field(512, ge=1, le=4096)
    temperature: float = Field(0.7, ge=0.1, le=2.0)
    top_p: float = Field(0.9, ge=0.0, le=1.0)
    top_k: int = Field(50, ge=1, le=100)
    repetition_penalty: float = Field(1.1, ge=1.0, le=2.0)

class GenerationResponse(BaseModel):
    id: str
    text: str
    usage: Dict[str, Any]
    created: int

class HealthResponse(BaseModel):
    status: str
    model_loaded: bool
    device: str
    loading_status: str
    loading_progress: int
    gpu_memory_used: float = 0.0
    gpu_memory_total: float = 0.0

In [7]:
# Cell 8: Create FastAPI App
model_manager = ModelManager()

app = FastAPI(title="Colab Model API", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/health", response_model=HealthResponse)
async def health_check():
    gpu_memory_used = 0
    gpu_memory_total = 0

    if torch.cuda.is_available():
        gpu_memory_used = torch.cuda.memory_allocated() / 1e9
        gpu_memory_total = torch.cuda.get_device_properties(0).total_memory / 1e9

    return HealthResponse(
        status="healthy" if model_manager.model is not None else "loading",
        model_loaded=model_manager.model is not None,
        device=str(model_manager.device) if model_manager.device else "unknown",
        loading_status=model_manager.loading_status,
        loading_progress=model_manager.loading_progress,
        gpu_memory_used=gpu_memory_used,
        gpu_memory_total=gpu_memory_total
    )

@app.post("/generate", response_model=GenerationResponse)
async def generate(request: GenerationRequest):
    if model_manager.model is None:
        raise HTTPException(status_code=503, detail=f"Model not ready. Status: {model_manager.loading_status}")

    try:
        result = model_manager.generate(
            prompt=request.prompt,
            max_tokens=request.max_tokens,
            temperature=request.temperature,
            top_p=request.top_p,
            top_k=request.top_k,
            repetition_penalty=request.repetition_penalty
        )

        return GenerationResponse(
            id=str(uuid.uuid4()),
            text=result['text'],
            usage=result['usage'],
            created=int(time.time())
        )
    except Exception as e:
        logger.error(f"Generation failed: {e}")
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/")
async def root():
    return {
        "message": "Model API running on Google Colab",
        "docs": "/docs",
        "health": "/health",
        "model_status": model_manager.loading_status
    }

In [8]:
# Cell 9: Load Model in Background
def load_model_in_background():
    logger.info("Starting model loading in background...")
    model_manager.load_model(use_4bit=True)

# Start loading
loading_thread = threading.Thread(target=load_model_in_background, daemon=True)
loading_thread.start()

In [9]:
# Cell 10: Start Server for Render Deployment
# For Render: Use environment variable PORT
import os

def start_server():
    # Render provides PORT environment variable
    port = int(os.environ.get("PORT", 8000))
    host = "0.0.0.0"

    logger.info(f"Starting server on {host}:{port}")

    # Run with uvicorn (for Render)
    uvicorn.run(app, host=host, port=port, log_level="info")

# For production on Render, you'd run this via the command:
# uvicorn main:app --host 0.0.0.0 --port $PORT

# For testing in Colab:
if __name__ == "__main__":
    port = 8000
    logger.info(f"Starting server on port {port}")

    def run_server():
        uvicorn.run(app, host="0.0.0.0", port=port, log_level="info")

    server_thread = threading.Thread(target=run_server, daemon=True)
    server_thread.start()
    time.sleep(3)
    print(f"✅ Server running on port {port}")
    print("📝 For Render deployment, see instructions below.")

INFO:     Started server process [3948]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ Server running on port 8000
📝 For Render deployment, see instructions below.


In [10]:
# Cell 11: Create Deployment Files for Render
# This cell creates the necessary files for Render deployment

# Create requirements.txt
requirements_content = """
torch>=2.0.0
torchvision>=0.15.0
torchaudio>=2.0.0
transformers>=4.35.0
accelerate>=0.24.0
bitsandbytes>=0.41.0
peft>=0.6.0
fastapi>=0.104.0
uvicorn>=0.24.0
python-multipart>=0.0.6
aiofiles>=23.2.0
python-dotenv>=1.0.0
sentencepiece>=0.1.99
protobuf>=3.20.0
huggingface-hub>=0.19.0
nest-asyncio>=1.5.0
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content.strip())
print("✅ Created requirements.txt")

# Create main.py for Render deployment
main_py_content = '''
import torch
import os
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import time
import uuid
from typing import Dict, Any
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Model Manager
class ModelManager:
    def __init__(self):
        self.model = None
        self.tokenizer = None
        self.device = None
        self.model_path = "/opt/render/project/src/models"
        self.loading_status = "not_started"

    def setup_device(self):
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
            logger.info(f"Using GPU: {torch.cuda.get_device_name(0)}")
        else:
            self.device = torch.device("cpu")
            logger.info("Using CPU")

    def load_model(self):
        try:
            self.loading_status = "loading"
            self.setup_device()

            logger.info("Loading tokenizer...")
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_path)
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token

            logger.info("Loading model...")
            if torch.cuda.is_available():
                self.model = AutoModelForCausalLM.from_pretrained(
                    self.model_path,
                    torch_dtype=torch.float16,
                    device_map="auto",
                    trust_remote_code=True
                )
            else:
                self.model = AutoModelForCausalLM.from_pretrained(
                    self.model_path,
                    torch_dtype=torch.float32,
                    trust_remote_code=True
                ).to(self.device)

            self.loading_status = "loaded"
            logger.info("✅ Model loaded successfully!")
            return True
        except Exception as e:
            self.loading_status = "failed"
            logger.error(f"Failed to load model: {e}")
            return False

    def generate(self, prompt: str, **kwargs):
        if self.model is None:
            raise Exception("Model not loaded")

        start_time = time.time()
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)

        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        else:
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=kwargs.get('max_tokens', 512),
                temperature=kwargs.get('temperature', 0.7),
                top_p=kwargs.get('top_p', 0.9),
                do_sample=True,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )

        generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
        generated_text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)

        return {"text": generated_text}

# Initialize model manager
model_manager = ModelManager()

# Create FastAPI app
app = FastAPI(title="Model API", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.on_event("startup")
async def startup_event():
    """Load model on startup"""
    import threading
    thread = threading.Thread(target=model_manager.load_model, daemon=True)
    thread.start()

@app.get("/health")
async def health_check():
    return {
        "status": "healthy" if model_manager.model is not None else "loading",
        "model_loaded": model_manager.model is not None,
        "device": str(model_manager.device) if model_manager.device else "unknown"
    }

@app.post("/generate")
async def generate(request: dict):
    if model_manager.model is None:
        raise HTTPException(status_code=503, detail="Model not ready")

    try:
        result = model_manager.generate(
            prompt=request.get("prompt", ""),
            max_tokens=request.get("max_tokens", 512),
            temperature=request.get("temperature", 0.7)
        )
        return {
            "id": str(uuid.uuid4()),
            "text": result["text"],
            "created": int(time.time())
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/")
async def root():
    return {"message": "Model API is running", "docs": "/docs"}

if __name__ == "__main__":
    port = int(os.environ.get("PORT", 8000))
    uvicorn.run(app, host="0.0.0.0", port=port)
'''

with open("main.py", "w") as f:
    f.write(main_py_content.strip())
print("✅ Created main.py")

# Create render.yaml for deployment configuration
render_yaml_content = """
services:
  - type: web
    name: model-api
    runtime: python
    plan: free  # or standard, professional
    buildCommand: pip install -r requirements.txt
    startCommand: uvicorn main:app --host 0.0.0.0 --port $PORT
    envVars:
      - key: PYTHON_VERSION
        value: 3.10.12
      - key: HF_TOKEN
        sync: false  # Set this in Render dashboard
    disk:
      name: model-storage
      mountPath: /opt/render/project/src/models
      sizeGB: 50  # Adjust based on model size
"""

with open("render.yaml", "w") as f:
    f.write(render_yaml_content.strip())
print("✅ Created render.yaml")

print("\n" + "="*60)
print("📦 DEPLOYMENT FILES CREATED SUCCESSFULLY!")
print("="*60)

✅ Created requirements.txt
✅ Created main.py
✅ Created render.yaml

📦 DEPLOYMENT FILES CREATED SUCCESSFULLY!


In [17]:
# Cell 12: Upload Model to Hugging Face (for Render to download)
# This creates a script to upload your model to Hugging Face

from huggingface_hub import HfApi
import os

# Set your Hugging Face username and model name
HF_USERNAME = "ugonna"
MODEL_NAME = "llama3.18B-Fine-tunedByUgo3"

# Path to your model files
MODEL_PATH = "/content/model-service/models"

# Login (make sure you have your token set)
from huggingface_hub import login
login(token=os.environ.get("HF_Token"))

# Create repository if it doesn't exist
api = HfApi()
try:
    api.create_repo(repo_id=f"{HF_USERNAME}/{MODEL_NAME}", exist_ok=True)
    print(f"✅ Repository created/exists: {HF_USERNAME}/{MODEL_NAME}")
except Exception as e:
    print(f"Repo creation error: {e}")

# Upload model files
api.upload_folder(
    folder_path=MODEL_PATH,
    repo_id=f"{HF_USERNAME}/{MODEL_NAME}",
    repo_type="model"
)
print("✅ Model uploaded successfully!")


print("📤 To upload your model to Hugging Face (so Render can download it):")
print("1. Run the script below after setting your HF_USERNAME and MODEL_NAME")
print("2. Make sure your model is in /content/model-service/models")

✅ Repository created/exists: ugonna/llama3.18B-Fine-tunedByUgo3


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../models/training_args.bin: 100%|##########| 5.69kB / 5.69kB            

  ...ice/models/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

  ...77991.672bf34e9b11.1650.0: 100%|##########| 14.0kB / 14.0kB            

  ...adapter_model.safetensors: 100%|#########9| 13.6MB / 13.6MB            

No files have been modified since last commit. Skipping to prevent empty commit.


✅ Model uploaded successfully!
📤 To upload your model to Hugging Face (so Render can download it):
1. Run the script below after setting your HF_USERNAME and MODEL_NAME
2. Make sure your model is in /content/model-service/models


In [12]:
# Cell 13: Instructions for Render Deployment
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                    🚀 RENDER DEPLOYMENT INSTRUCTIONS 🚀                       ║
╚══════════════════════════════════════════════════════════════════════════════╝

STEP 1: Prepare Your Model for Deployment
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Option A: Upload model to Hugging Face (Recommended)
  • Upload your model to Hugging Face Hub using the script in Cell 12
  • Make sure it's publicly accessible or use a token

Option B: Use direct download in Render
  • Your model will download on Render during build (slower startup)

STEP 2: Set Up Render Account
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  • Go to https://render.com
  • Sign up/Login with GitHub
  • Click "New +" → "Web Service"

STEP 3: Configure the Web Service
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  • Connect your GitHub repository
  • Name: model-api
  • Runtime: Python
  • Build Command: pip install -r requirements.txt
  • Start Command: uvicorn main:app --host 0.0.0.0 --port $PORT
  • Plan: Free (1GB RAM, 0.1 CPU) or Professional (for better performance)

STEP 4: Add Environment Variables
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Under "Environment Variables", add:

  Key: HF_TOKEN
  Value: your_hugging_face_token (if model is private)

  Key: PYTHON_VERSION
  Value: 3.10.12

STEP 5: Add Persistent Disk (Important!)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  • Go to your service → Settings → Disks
  • Add Disk:
    • Name: model-storage
    • Mount Path: /opt/render/project/src/models
    • Size: 50 GB (or size of your model + buffer)

STEP 6: Deploy
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  • Click "Create Web Service"
  • Wait for build and deployment (may take 10-30 minutes)
  • Your API will be available at: https://your-service-name.onrender.com

STEP 7: Test Your Deployed API
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  • Health Check: curl https://your-service.onrender.com/health
  • Generate: curl -X POST https://your-service.onrender.com/generate \\
      -H "Content-Type: application/json" \\
      -d '{"prompt": "Hello, world!", "max_tokens": 100}'

IMPORTANT NOTES:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ⚠️ Free tier limitations:
     • 1GB RAM (may not be enough for 8B model)
     • 0.1 CPU (very slow inference)
     • Spins down after 15 minutes of inactivity (cold start)

  ✅ For better performance (8B model with 4-bit quantization needs ~4GB RAM):
     • Use Professional tier ($7/month) with 2.5GB RAM
     • Or use Starter tier ($0/month) with limited resources

  🔄 Keep service alive (prevent spin-down):
     • Use uptime monitoring service like UptimeRobot
     • Ping the /health endpoint every 10 minutes

  💾 Persistent Disk:
     • Model will remain on disk between deployments
     • First deployment downloads model, subsequent deploys use cached model
""")


╔══════════════════════════════════════════════════════════════════════════════╗
║                    🚀 RENDER DEPLOYMENT INSTRUCTIONS 🚀                       ║
╚══════════════════════════════════════════════════════════════════════════════╝

STEP 1: Prepare Your Model for Deployment
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Option A: Upload model to Hugging Face (Recommended)
  • Upload your model to Hugging Face Hub using the script in Cell 12
  • Make sure it's publicly accessible or use a token

Option B: Use direct download in Render
  • Your model will download on Render during build (slower startup)

STEP 2: Set Up Render Account
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  • Go to https://render.com
  • Sign up/Login with GitHub
  • Click "New +" → "Web Service"

STEP 3: Configure the Web Service
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  • Connect your GitHub reposit

In [13]:
# Cell 15: Test Local API (Before Deployment)
import requests
import time

def test_local_api():
    """Test the local API before deploying"""
    base_url = "http://localhost:8000"

    print("Testing local API...")

    # Wait for model to load
    for i in range(60):
        try:
            response = requests.get(f"{base_url}/health", timeout=5)
            if response.status_code == 200:
                health = response.json()
                if health['model_loaded']:
                    print("✅ Model is ready!")
                    break
        except:
            pass
        time.sleep(2)

    # Test generation
    try:
        response = requests.post(
            f"{base_url}/generate",
            json={"prompt": "Explain AI in simple terms:", "max_tokens": 100},
            timeout=60
        )
        if response.status_code == 200:
            result = response.json()
            print(f"\nGenerated: {result['text'][:200]}...")
            print("✅ Local API test successful!")
        else:
            print(f"Error: {response.status_code}")
    except Exception as e:
        print(f"Error: {e}")

# Uncomment to test:
test_local_api()

Testing local API...
INFO:     127.0.0.1:52434 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:52442 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:52446 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:52454 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:52466 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:41124 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:41128 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:41138 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:41152 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:41164 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:58026 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:58042 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:58056 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:58062 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:58064 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:33966 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:33980 - "GET /health HTTP/1.1" 